In [ ]:
# Importanciónes y dependencias
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Configuraciones de rutas (pathlib resuelve conflictos rutas linux y windows)
BASE_PATH = Path().resolve()
DATA_PROCESSED = BASE_PATH / ".." / "data" / "processed"
DATA_CURATED = BASE_PATH / ".." / "data" / "curated"

DATA_CURATED.mkdir(parents=True, exist_ok=True)

file_path = DATA_PROCESSED / "gaia_neighbor_processed.csv"

In [8]:
# Lectura del arcchivo csv rawprocessed
df = pd.read_csv(file_path)

# Informacion del dataframe
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

# Impresion de primeras filas
df.head()

Filas: 102625
Columnas: 8


,source_id,ra,dec,parallax,parallax_over_error,phot_g_mean_mag,bp_rp,distance_pc
0,1495713771471104,42.344198,2.290799,18.096821,823.04285,12.842120,2.263671,55.258325
1,2183668748454272,44.753144,3.972805,18.404966,697.28980,14.311949,2.499146,54.333162
2,3656185400409472,46.965442,5.236058,19.264496,388.73822,15.862782,3.269577,51.908963
3,6347480627407104,42.935498,5.651869,17.650109,1014.52515,11.135140,1.621806,56.656875
4,6397856299125120,43.343943,6.132225,22.178355,451.88208,15.695479,3.317692,45.089007


In [ ]:
# Calcular Magnitud Absoluta
df['abs_mag'] = df['phot_g_mean_mag'] - 5 * np.log10(df['distance_pc']) + 5

df.head()

,source_id,ra,dec,parallax,parallax_over_error,phot_g_mean_mag,bp_rp,distance_pc,abs_mag
0,1495713771471104,42.344198,2.290799,18.096821,823.04285,12.842120,2.263671,55.258325,9.130131
1,2183668748454272,44.753144,3.972805,18.404966,697.28980,14.311949,2.499146,54.333162,10.636624
2,3656185400409472,46.965442,5.236058,19.264496,388.73822,15.862782,3.269577,51.908963,12.286571
3,6347480627407104,42.935498,5.651869,17.650109,1014.52515,11.135140,1.621806,56.656875,7.368877
4,6397856299125120,43.343943,6.132225,22.178355,451.88208,15.695479,3.317692,45.089007,12.425126


In [10]:
# Clasificar por tipo espectral
def classify_spectral(color):
    if color < -0.3: return 'O'
    if color < 0.0: return 'B'
    if color < 0.3: return 'A'
    if color < 0.8: return 'F'
    if color < 1.15: return 'G'
    if color < 1.6: return 'K'
    return 'M'

df['spectral_type'] = df['bp_rp'].apply(classify_spectral)

df.head()

,source_id,ra,dec,parallax,parallax_over_error,phot_g_mean_mag,bp_rp,distance_pc,abs_mag,spectral_type
0,1495713771471104,42.344198,2.290799,18.096821,823.04285,12.842120,2.263671,55.258325,9.130131,M
1,2183668748454272,44.753144,3.972805,18.404966,697.28980,14.311949,2.499146,54.333162,10.636624,M
2,3656185400409472,46.965442,5.236058,19.264496,388.73822,15.862782,3.269577,51.908963,12.286571,M
3,6347480627407104,42.935498,5.651869,17.650109,1014.52515,11.135140,1.621806,56.656875,7.368877,M
4,6397856299125120,43.343943,6.132225,22.178355,451.88208,15.695479,3.317692,45.089007,12.425126,M


In [ ]:
# Estimación de Masa (esta es una aproximación para Secuencia Principal)
df['estimated_mass'] = 10**((4.75 - df['abs_mag']) / 10)

df.head()

,source_id,ra,dec,parallax,parallax_over_error,phot_g_mean_mag,bp_rp,distance_pc,abs_mag,spectral_type,estimated_mass
0,1495713771471104,42.344198,2.290799,18.096821,823.04285,12.842120,2.263671,55.258325,9.130131,M,0.364743
1,2183668748454272,44.753144,3.972805,18.404966,697.28980,14.311949,2.499146,54.333162,10.636624,M,0.257832
2,3656185400409472,46.965442,5.236058,19.264496,388.73822,15.862782,3.269577,51.908963,12.286571,M,0.176337
3,6347480627407104,42.935498,5.651869,17.650109,1014.52515,11.135140,1.621806,56.656875,7.368877,M,0.547157
4,6397856299125120,43.343943,6.132225,22.178355,451.88208,15.695479,3.317692,45.089007,12.425126,M,0.170800


In [12]:
# Flag "Parecida al Sol" (Tipo G y Magnitud Absoluta entre 4 y 5.5)
df['is_sun_like'] = (df['spectral_type'] == 'G') & (df['abs_mag'].between(4, 5.5))

df.head()

,source_id,ra,dec,parallax,parallax_over_error,phot_g_mean_mag,bp_rp,distance_pc,abs_mag,spectral_type,estimated_mass,is_sun_like
0,1495713771471104,42.344198,2.290799,18.096821,823.04285,12.842120,2.263671,55.258325,9.130131,M,0.364743,False
1,2183668748454272,44.753144,3.972805,18.404966,697.28980,14.311949,2.499146,54.333162,10.636624,M,0.257832,False
2,3656185400409472,46.965442,5.236058,19.264496,388.73822,15.862782,3.269577,51.908963,12.286571,M,0.176337,False
3,6347480627407104,42.935498,5.651869,17.650109,1014.52515,11.135140,1.621806,56.656875,7.368877,M,0.547157,False
4,6397856299125120,43.343943,6.132225,22.178355,451.88208,15.695479,3.317692,45.089007,12.425126,M,0.170800,False


In [14]:
# Selección de variables para archivo curated
columns_to_keep = [
    'source_id', 'abs_mag', 'bp_rp', 
    'spectral_type', 'distance_pc', 
    'estimated_mass', 'is_sun_like'
]

df_stars_type_curated = df[columns_to_keep].copy()

# Guardado final
output_path = DATA_CURATED / "star_spectral_curated.csv"
df_stars_type_curated.to_csv(output_path, index=False)

print(f"Archivo Gaia guardado en: {output_path}")
print(f"Resumen: {len(df_stars_type_curated)} estrellas procesadas.")

Archivo Gaia guardado en: F:\Usuario\Diego\Estudios\Ilerna\EspacializacionBigDataIA\ProyectoBigData\proyectoBCSS\notebook\..\data\curated\star_spectral_curated.csv
Resumen: 102625 estrellas procesadas.
